## GoogLeNet with Galaxy10 DECals Dataset

The Galaxy10 DECals dataset consists of 17,736 256 $\times$ 256 images of galaxies classified into 10 categories. The underlying images were originally sourced from the _Sloan Digital Sky Survey (SDSS)_ via the Galaxy Zoo project. Here we use the dataset to train a galaxy type classifier based on GoogLeNet.

In [ ]:
import torch
import torchvision
import torchvision.transforms.functional as TF
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
import numpy as np
from tqdm.notebook import trange
import h5py
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load the Data

In [ ]:
with h5py.File('Galaxy10_DECals.h5', 'r') as gF:
    images = np.array(gF['images'])
    labels = np.array(gF['ans'])

In [ ]:
from sklearn.utils import shuffle
images, labels = shuffle(images, labels)

### Prepare the Data

The data must be prepared in a similar fashion to the MNIST examples except now there are three channels instead of one. The classes must be converted to a categorical variable.

In [ ]:
train_idx, test_idx = train_test_split(np.arange(labels.shape[0]), test_size=0.1)
X_train, y_train, X_test, y_test = images[train_idx], labels[train_idx], images[test_idx], labels[test_idx]

encoder = OneHotEncoder(sparse_output=False)
train_y = encoder.fit_transform(y_train.reshape(-1, 1))
test_y = encoder.transform(y_test.reshape(-1, 1))

X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train /= 255
X_test /= 255

X_train = torch.Tensor(X_train).permute(0, 3, 1, 2)
train_y = torch.Tensor(train_y)
train_dataset = TensorDataset(X_train, train_y)

X_test = torch.Tensor(X_test).permute(0, 3, 1, 2)
test_y = torch.Tensor(test_y)
test_dataset = TensorDataset(X_test, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, 
                                          shuffle=False, drop_last=True)

In [ ]:
class CustomAugmentations:
    def __init__(self, rotation_range, width_shift_range, height_shift_range, shear_range, zoom_range, horizontal_flip):
        self.rotation_range = rotation_range
        self.width_shift_range = width_shift_range
        self.height_shift_range = height_shift_range
        self.shear_range = shear_range
        self.zoom_range = zoom_range
        self.horizontal_flip = horizontal_flip

    def __call__(self, img):
        # Random rotation
        angle = torch.empty(1).uniform_(-self.rotation_range, self.rotation_range).item()
        img = TF.rotate(img, angle, interpolation=TF.InterpolationMode.NEAREST)

        # Random width shift
        width_shift = torch.empty(1).uniform_(-self.width_shift_range, self.width_shift_range).item()
        img = TF.affine(img, angle=0, translate=(width_shift * img.shape[2], 0), scale=1, shear=0, interpolation=TF.InterpolationMode.NEAREST)

        # Random height shift
        height_shift = torch.empty(1).uniform_(-self.height_shift_range, self.height_shift_range).item()
        img = TF.affine(img, angle=0, translate=(0, height_shift * img.shape[1]), scale=1, shear=0, interpolation=TF.InterpolationMode.NEAREST)

        # Random shear
        shear = torch.empty(1).uniform_(-self.shear_range, self.shear_range).item()
        img = TF.affine(img, angle=0, translate=(0, 0), scale=1, shear=shear, interpolation=TF.InterpolationMode.NEAREST)

        # Random zoom
        zoom = torch.empty(1).uniform_(1 - self.zoom_range, 1 + self.zoom_range).item()
        img = TF.affine(img, angle=0, translate=(0, 0), scale=zoom, shear=0, interpolation=TF.InterpolationMode.NEAREST)

        # Random horizontal flip
        if self.horizontal_flip and torch.rand(1).item() > 0.5:
            img = TF.hflip(img)
        
        return img

custom_transforms = transforms.Compose([
    CustomAugmentations(rotation_range=40, 
                        width_shift_range=0.2, 
                        height_shift_range=0.2, 
                        shear_range=0.2, 
                        zoom_range=0.2, 
                        horizontal_flip=True)
])

class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

aug_train_dataset = AugmentedDataset(train_dataset, 
                                     transform=custom_transforms)

train_loader = DataLoader(aug_train_dataset, batch_size=128, shuffle=True)

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs, device):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            # Forward pass
            outputs, aux_out1, aux_out2 = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            loss_aux1 = loss_fn(aux_out1.squeeze(), batch_y)
            loss_aux2 = loss_fn(aux_out2.squeeze(), batch_y)
            tot_loss = loss + loss_aux1 + loss_aux2
            
            # Backward pass and optimization
            optimizer.zero_grad()
            tot_loss.backward()
            optimizer.step()

            train_loss += tot_loss.item() * batch_X.size(0)
            
            # Calculate accuracy
            predicted = torch.argmax(outputs.squeeze(), dim=1)
            targets = torch.argmax(batch_y, dim=1)
            correct_train += (predicted == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)
                # Calculate accuracy
                predicted = torch.argmax(outputs.squeeze(), dim=1)
                targets = torch.argmax(batch_y, dim=1)
                correct_test += (predicted == targets).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, \
            Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
        
    return history

### Build GoogLeNet Model

Define a function that will create an Inception Block.

In [ ]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, c1, c2_1, c2_2, c3_1, c3_2, c4):
        super(InceptionBlock, self).__init__()
        self.branch1 = nn.Conv2d(in_channels, c1, kernel_size=1, padding=0)
        
        self.branch2_1 = nn.Conv2d(in_channels, c2_1, kernel_size=1, padding=0)
        self.branch2_2 = nn.Conv2d(c2_1, c2_2, kernel_size=3, padding=1)
        
        self.branch3_1 = nn.Conv2d(in_channels, c3_1, kernel_size=1, padding=0)
        self.branch3_2 = nn.Conv2d(c3_1, c3_2, kernel_size=5, padding=2)
        
        self.branch4_pool = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.branch4_1 = nn.Conv2d(in_channels, c4, kernel_size=1, padding=0)
        
    def forward(self, x):
        branch1 = F.relu(self.branch1(x))
        
        branch2 = F.relu(self.branch2_1(x))
        branch2 = F.relu(self.branch2_2(branch2))
        
        branch3 = F.relu(self.branch3_1(x))
        branch3 = F.relu(self.branch3_2(branch3))
        
        branch4 = self.branch4_pool(x)
        branch4 = F.relu(self.branch4_1(branch4))
        
        outputs = [branch1, branch2, branch3, branch4]
        return torch.cat(outputs, 1)

In [ ]:
class AuxiliaryClassifier(nn.Module):
    def __init__(self, in_channels, num_classes=10):
        super(AuxiliaryClassifier, self).__init__()
        self.pool = nn.AvgPool2d(kernel_size=5, stride=3)
        self.conv = nn.Conv2d(in_channels, 128, kernel_size=1, padding=0)
        self.fc1 = nn.Linear(128 * 4 * 4, 1024)
        self.dropout = nn.Dropout(0.7)
        self.fc2 = nn.Linear(1024, num_classes)
    
    def forward(self, x):
        x = self.pool(x)
        x = F.relu(self.conv(x))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [ ]:
class GoogLeNet(nn.Module):
    def __init__(self, num_classes=10):
        super(GoogLeNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.maxpool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        
        self.conv2_1 = nn.Conv2d(64, 64, kernel_size=1, padding=0)
        self.conv2_2 = nn.Conv2d(64, 192, kernel_size=3, padding=1)
        self.maxpool2 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(192)
        
        self.inception3a = InceptionBlock(192, 64, 96, 128, 16, 32, 32)
        self.inception3b = InceptionBlock(256, 128, 128, 192, 32, 96, 64)
        self.maxpool3 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.inception4a = InceptionBlock(480, 192, 96, 208, 16, 48, 64)
        self.aux1 = AuxiliaryClassifier(512, num_classes)
        
        self.inception4b = InceptionBlock(512, 160, 112, 224, 24, 64, 64)
        self.inception4c = InceptionBlock(512, 128, 128, 256, 24, 64, 64)
        self.inception4d = InceptionBlock(512, 112, 144, 288, 32, 64, 64)
        self.aux2 = AuxiliaryClassifier(528, num_classes)
        
        self.inception4e = InceptionBlock(528, 256, 160, 320, 32, 128, 128)
        self.maxpool4 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.inception5a = InceptionBlock(832, 256, 160, 320, 32, 128, 128)
        self.inception5b = InceptionBlock(832, 384, 192, 384, 48, 128, 128)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(1024, num_classes)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.maxpool1(x)
        x = self.bn1(x)
        
        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = self.maxpool2(x)
        x = self.bn2(x)
        
        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool3(x)
        
        x = self.inception4a(x)
        aux1 = self.aux1(x) if self.training else None
        
        x = self.inception4b(x)
        x = self.inception4c(x)
        x = self.inception4d(x)
        aux2 = self.aux2(x) if self.training else None
        
        x = self.inception4e(x)
        x = self.maxpool4(x)
        
        x = self.inception5a(x)
        x = self.inception5b(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        
        if self.training:
            return x, aux1, aux2
        else:
            return x

In [ ]:
googlenet_model = GoogLeNet().to(device)

In [ ]:
# Define the optimizer (Adam in this case)
optimizer = optim.Adam(googlenet_model.parameters(), lr=0.001)

# Define the loss function (categorical cross-entropy in this case)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
no_epochs = 50

In [ ]:
history_dict = dict()

### Train a model with Data Aumentation

In [ ]:
history = train_model(googlenet_model, train_loader, test_loader,
                      loss_fn, optimizer, no_epochs, device)

### Plot Validation Accuracy

In [ ]:
epochs = range(1, no_epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
val_acc_values = history['test_acc']
ax.plot(epochs, val_acc_values, label = 'GoogLeNet Validation')
acc_values = history['train_acc']
ax.plot(epochs, acc_values, label = 'GoogLeNet Training')
    
ax.set_xlabel('Epochs')
ax.set_ylabel('Accuracy')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('TestGoogLeNet.png', dpi=300, bbox_inches='tight')